# Week 2 — Preprocess images + baseline Tesseract gap analysis

This notebook replaces the handout's grayscale -> squash-resize -> hard-threshold
pipeline with `src/preprocessing.py`'s deskew + adaptive-contrast + letterbox
pipeline (see that file's docstring for *why* — short version: a fixed-size
squash and a global binary threshold both destroy information a transformer
OCR model needs). It then runs the same Tesseract baseline the handout asks
for, but scores it with CER/WER instead of eyeballing 5 outputs.


In [ ]:
# Run once per Colab session (safe to re-run: pulls latest instead of
# silently reusing a stale checkout from an earlier run this session).
import os
import subprocess
import sys

REPO_DIR = '/content/repo'
# IMPORTANT: pinned to this specific branch, not the repo's default branch
# -- see Quickstart_Colab.ipynb's Cell 1 for why (an unqualified clone/pull
# can silently track a branch that's missing later fixes, and `git pull`
# then reports "Already up to date" against the wrong branch with no
# warning). Replace with your own fork's URL/branch if running elsewhere,
# or with 'main' once this branch has actually been merged there.
BRANCH = 'claude/high-accuracy-solution-r9r2t1'
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} --single-branch https://github.com/Rameesha-Malik/ML-SI-Solution-Urdu-OCR.git {REPO_DIR}
else:
    print(f'{REPO_DIR} already exists -- fetching + resetting to origin/{BRANCH} instead of re-cloning.')
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{BRANCH}

os.chdir(REPO_DIR)
!git -C {REPO_DIR} log -1 --format='Repo HEAD: %h %s'

# Deliberately NOT --upgrade-strategy eager, and NOT --force-reinstall on
# the whole file: both would make pip consider reinstalling `torch` even
# though Colab's pre-installed torch already satisfies our ">=2.2.0" bound
# -- and a freshly pip-resolved torch build was observed, in testing, to be
# missing a CUDA sub-package it needs (ImportError: libcublas.so.13), which
# would trade a fixable Python-level bug for a broken GPU runtime. Plain
# `--upgrade -r requirements.txt` still fully resolves and changes
# transformers/pillow/gradio (their pins are strict enough that pip must
# change them regardless of strategy -- upgrade-strategy only affects
# whether *unnamed transitive* dependencies like torch get proactively
# touched), while leaving torch itself alone.
!pip install -q --upgrade -r requirements.txt

# Ground-truth check of what's on disk, via a subprocess (`pip show`) --
# NOT `import transformers`/`import PIL`. If a notebook cell earlier in
# this session already imported one of these, Python caches that module
# in memory for the rest of the process; reinstalling on disk does not
# change what's already loaded there. The actual training/eval scripts
# run as a separate `!python ...` subprocess each time, so they always
# see what's genuinely on disk -- this mirrors that rather than this
# kernel's own (possibly stale) view.
def _installed_version(pkg):
    result = subprocess.run(['pip', 'show', pkg], capture_output=True, text=True)
    return next(
        (l.split(':', 1)[1].strip() for l in result.stdout.splitlines() if l.startswith('Version:')),
        '?',
    )

versions = {pkg: _installed_version(pkg) for pkg in ('transformers', 'pillow', 'gradio')}
print('Installed on disk:', versions)

if 'src' not in sys.path:
    sys.path.insert(0, 'src')

expected_major = {'transformers': ('4.',), 'pillow': ('11.',), 'gradio': ('4.', '5.', '6.0', '6.1')}
bad = {
    p: v for p, v in versions.items()
    if not any(v.startswith(prefix) for prefix in expected_major[p])
}
if bad:
    raise RuntimeError(
        f'Unexpected version(s) on disk: {bad} (expected transformers 4.x, '
        'pillow 11.x, gradio <6.20 per requirements.txt). pip could not '
        'fully satisfy the pins here -- likely another package in this '
        'Colab image forces a conflicting version. Re-run this cell and '
        'share the FULL pip output above (not just this message) so the '
        'real conflicting package can be identified.'
    )

# Functional smoke test, not just a version check -- and run in a genuinely
# fresh subprocess, not this notebook kernel. Pillow (and any package with a
# compiled C extension) cannot have its native .so swapped out in an
# already-running process even after a correct on-disk reinstall; the only
# way to know it will actually work is to check from a brand new process,
# which is also exactly what the training/eval scripts do.
pil_check = subprocess.run(
    ['python3', '-c',
     "from PIL import Image; im = Image.new('RGB', (32, 32), 'white'); "
     "im.save('/tmp/_pil_smoke_test.png'); "
     "Image.open('/tmp/_pil_smoke_test.png').load(); print('PIL round-trip OK')"],
    capture_output=True, text=True,
)
print(pil_check.stdout.strip())
if pil_check.returncode != 0:
    raise RuntimeError(
        'Pillow is installed but broken (fails to save/load an image even in '
        'a fresh process) -- this is not a version-pin issue, it is a '
        'corrupted install, most likely leftover mixed files on disk from '
        'earlier pip installs/reinstalls in this same Colab VM. '
        '`Runtime > Restart session` will NOT fix this (it only restarts '
        'the Python process, not the disk). You need '
        '`Runtime > Disconnect and delete runtime`, then reconnect to a '
        'brand new VM and run this notebook from Cell 1 on that clean '
        'slate.\n\nOriginal error:\n' + pil_check.stderr[-1500:]
    )

print('Setup complete -- all pinned versions correctly installed on disk and working.')


## Part A — Preprocess every collected image

In [ ]:
from preprocessing import preprocess_folder, PreprocessConfig

count = preprocess_folder('data/raw', 'data/processed', PreprocessConfig())
print(f'Preprocessed {count} images -> data/processed/')


In [ ]:
# Visual sanity check: compare a few raw vs. processed images.
import glob
from PIL import Image
import matplotlib.pyplot as plt

raw_samples = sorted(glob.glob('data/raw/**/*.png', recursive=True))[:3] + \
              sorted(glob.glob('data/raw/**/*.jpg', recursive=True))[:3]
fig, axes = plt.subplots(len(raw_samples[:3]), 2, figsize=(10, 3 * len(raw_samples[:3])))
for i, raw_path in enumerate(raw_samples[:3]):
    import os
    processed_path = os.path.join('data/processed', os.path.basename(raw_path))
    axes[i, 0].imshow(Image.open(raw_path)); axes[i, 0].set_title('raw'); axes[i, 0].axis('off')
    if os.path.exists(processed_path):
        axes[i, 1].imshow(Image.open(processed_path)); axes[i, 1].set_title('processed'); axes[i, 1].axis('off')
plt.tight_layout()
plt.show()


## Part B — Baseline Tesseract OCR + quantified gap analysis

Install Tesseract's Urdu language pack first (Colab shell command):

In [ ]:
!apt-get -qq install -y tesseract-ocr tesseract-ocr-urd
!pip install -q pytesseract

In [ ]:
import subprocess
result = subprocess.run(
    ['python', 'scripts/tesseract_baseline.py',
     '--labels-csv', 'data/labels.csv',
     '--report-csv', 'reports/tesseract_baseline.csv'],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


## Why We Need a Better Model

*(copy this section, filled in with your printed CER%/examples above, into your GitHub README)*

Tesseract fails on Urdu because... its Urdu language model was trained on a much
smaller and less varied dataset than its English one, and it does not handle
Nastaliq's diagonal letter-stacking and heavy contextual joining well — it
tends to segment the image into a flat left-to-right strip of isolated glyph
guesses, which breaks down badly whenever letters overlap vertically or a
word's letterforms change with position. The character error rate printed
above (see `reports/tesseract_baseline.csv` for every example) quantifies
just how large that gap is, which is exactly the gap Weeks 3-4's fine-tuned
model is built to close.
